In [17]:
# ------------------------------------------
# modulos pip
# ------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [18]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
#  Import da base "incidentes-regras-rt"

df_incidentes_o = pd.read_csv('/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/incidentes-regras-rt-def.csv')

In [20]:
#  Drop de colunas irrelevantes para o modelo

df_incidentes = df_incidentes_o.drop(columns=['ORDER_ID', 'INCIDENT_ID', 'DATE', 'DISCIPLINE_RULE_BUCKET', 'NAME']) 

In [21]:
# Criação de três novas colunas de acordo com o tipo de punição sofrida (permanent block, temporary block, warning)

df_incidentes = pd.get_dummies(df_incidentes, columns=['PUNISHMENT_TYPE'])

In [22]:
#  Criação de sete novas colunas de acordo com a categoria da regra infringida no incidente
# (covid, other, discipline, fraud, manual, performance, warning)

df_incidentes = pd.get_dummies(df_incidentes, columns=['CATEGORY_RULE'])

In [23]:
#  Análise quantitativa das novas colunas criadas de acordo com a categoria de regra

df_incidentes[['CATEGORY_RULE_Covid', 'CATEGORY_RULE_Other', 'CATEGORY_RULE_Discipline', 'CATEGORY_RULE_Fraud', 'CATEGORY_RULE_Manual', 'CATEGORY_RULE_Performance', 'CATEGORY_RULE_Warning']].sum()

CATEGORY_RULE_Covid              476
CATEGORY_RULE_Other                4
CATEGORY_RULE_Discipline     2217495
CATEGORY_RULE_Fraud            71962
CATEGORY_RULE_Manual           27200
CATEGORY_RULE_Performance       2902
CATEGORY_RULE_Warning          85562
dtype: int64

In [24]:
#  Drop das colunas de categoria "covid" e "outros"
#  Números desprezíveis quando comparados ao total das outras categorias
# e ao total de IDs da base consolidada
 
df_incidentes = df_incidentes.drop(columns=['CATEGORY_RULE_Covid', 'CATEGORY_RULE_Other'])

In [25]:
#  Adequação do nome das colunas (ID: compatibilidade da chave com a base consolidadada)

df_incidentes = df_incidentes.rename(columns={'STOREKEEPER_ID': 'ID', 'PUNISHMENT_TYPE_permanent_block': 'PERMANENT_BLOCK', 'PUNISHMENT_TYPE_temporary_block': 'TEMPORARY_BLOCKS', 'PUNISHMENT_TYPE_warning': 'WARNINGS', 'CATEGORY_RULE_Discipline' : 'DISCIPLINE_INCIDENTS', 'CATEGORY_RULE_Fraud' : 'FRAUD_INCIDENTS', 'CATEGORY_RULE_Manual' : 'MANUAL_INCIDENTS', 'CATEGORY_RULE_Performance' : 'PERFORMANCE_INCIDENTS', 'CATEGORY_RULE_Warning' : 'WARNING_INCIDENTS'})

In [26]:
#  Verificação da quantidade de incidentes registrados na base

df_incidentes.shape

(2405601, 10)

In [27]:
#  CRIAÇÃO DA FEATURE
# ---------------------------------------------------------------------------------
#  Soma do número de incidentes - de acordo com o tipo de punição e o tipo de regra
# envolvida - de cada entregador

df_incidentes = df_incidentes.groupby(by=['ID']).sum().reset_index()

In [28]:
df_incidentes

,ID,PUNISHMENT_MINUTES,PERMANENT_BLOCK,TEMPORARY_BLOCKS,WARNINGS,DISCIPLINE_INCIDENTS,FRAUD_INCIDENTS,MANUAL_INCIDENTS,PERFORMANCE_INCIDENTS,WARNING_INCIDENTS
0,32799,7200,0.0,5.0,0.0,0.0,5.0,0.0,0.0,0.0
1,32825,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,32924,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,32932,360,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,33027,21600000,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
185915,1558561,15,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
185916,1558691,15,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
185917,1558770,15,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
185918,1558815,15,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


In [29]:
df_incidentes['ID'].nunique()

185920